## ppd

In [ ]:
USE [SAMANTHA]
GO
If object_id('tmp_gestion_diners_ppd_300') is not null    
Drop table tmp_gestion_diners_ppd_300;    
If object_id('cont_gestion_diners_ppd') is not null    
Drop table cont_gestion_diners_ppd;    
If object_id('tmp_gestion_diners_ppd_301') is not null    
Drop table tmp_gestion_diners_ppd_301;    
If object_id('tmp_gestion_diners_ppd_302') is not null    
Drop table tmp_gestion_diners_ppd_302;    
If object_id('tmp_gestion_diners_ppd_303') is not null    
Drop table tmp_gestion_diners_ppd_303;    
If object_id('tmp_gestion_diners_ppd_304') is not null    
Drop table tmp_gestion_diners_ppd_304;    
If object_id('vnts_diners_1') is not null     
drop table vnts_diners_1;    
/*=================================================================================================================*/
DECLARE @FECHAINI date = '2026-07-13';

DECLARE @PERIODO varchar(7) = FORMAT(@FECHAINI, 'yyyy-MM');

DECLARE @CAMPANA  varchar(50) = @PERIODO + ' DINERS';
DECLARE @CAMPANA1 varchar(50) = @PERIODO + ' DINERS PLUS';
DECLARE @CAMPANA2 varchar(50) = @PERIODO + ' DINERS DE';
DECLARE @CAMPANA3 varchar(50) = @PERIODO + ' DINERS CD';
    
  
    
--SET @FECHAINI= CAST(@AÑO AS NvarChar(4))+'-'+CAST(@MES AS NvarChar(2))+'-'+CAST(@DIA AS NvarChar(2))    
/*=================================================================================================================*/    
    
Select dni+' '+PHONE_NUMBER AS Llave_Primaria,* Into tmp_gestion_diners_ppd_300   
From [SAMANTHA].[dbo].[tmp_llamadas_mes]    
where Fecha_Llamada=@FECHAINI    
and Nombre_Campana IN (@CAMPANA,@CAMPANA1,@CAMPANA2,@CAMPANA3)

--select * from tmp_gestion_diners_ppd_300
/*=================================================================================================================*/    
    
SELECT DNI,COUNT(*) AS CANT INTO cont_gestion_diners_ppd FROM tmp_gestion_diners_ppd_300 --where not DNI_Ejecutivo='VDAD'     
GROUP BY DNI    
/*=================================================================================================================*/    
    
Select     
C.FECHA_ENVIO,    
Case     
When a.Nombre_Campana IN  (@CAMPANA,@CAMPANA1,@CAMPANA2,@CAMPANA3) Then 'DINERS PPD PLUS' Else 'MAL' End As Servicio,    
convert(varchar,C.SERVICIO)+' '+a.Dni+' '+Convert(Varchar,a.Fecha_Llamada,103) as Llave_X_Dia,    
convert(varchar,C.SERVICIO)+' '+a.Dni as Llave_X_Mes,    
a.Dni As Dni_Cliente,    
a.DNI_Ejecutivo,    
a.Ejecutivo,    
a.Fecha_Hora_Llamada,    
a.Fecha_Llamada,    
a.Trama_Hora,    
a.PHONE_NUMBER as Telefono_Llamado,    
b.[NIVEL 2] AS Estados,    
b.[NIVEL 3] AS  Sub_estado,    
b.[NIVEL 4] AS Descripcion,    
ISNULL(C.MARCA,'SP') as [ESTADO VALIDACION],    
Case When b.[NIVEL 3] = 'VENTA' Then 1 Else '' End AS Ventas,    
null AS Monto_Abonar,    
a.Codigo_Paleta,    
b.PESO as Pesos,    
c.prioridad_inicial,    
b.cod_result,    
1 as Cantidad,    
b.cod_result AS INDICE,    
[NIVEL 2]+'>>'+b.[NIVEL 3]+'>>'+b.[NIVEL 4] AS data_banco,    
CONVERT(nvarchar(8),DATEADD(SECOND,segundos,0),114) as [TMO],    
NULL as OBSERVACIONES,    
C.MPPD /*CASE WHEN C.MARCA='PLUS' THEN 'PPD PLUS' ELSE C.MARCA/*(ISNULL(c.MPPD,'')+' '+ISNULL(c.MDS,'')+' '+ISNULL(c.MBT,''))*/ END*/  AS BASE,    
a.Inicio, a.Fin    
Into tmp_gestion_diners_ppd_301    
from tmp_gestion_diners_ppd_300 a     
LEFT Join ODIN..tTipologia_Diners_PPD b on a.Codigo_Paleta =b.cod COLLATE database_default    
LEFT Join ODIN..Base_Maestra_Diners_Vigente C on a.Dni  =C.[NumDoc] COLLATE database_default 
and (case when a.Nombre_Campana=@CAMPANA then 'PPD' WHEN a.Nombre_Campana=@CAMPANA1 THEN 'PPD PLUS' WHEN a.Nombre_Campana=@CAMPANA2 THEN 'DE' WHEN a.Nombre_Campana=@CAMPANA3 THEN 'CD' END)=C.MPPD 
where C.SERVICIO in('01','1') /*and not a.DNI_Ejecutivo='VDAD'*/ and c.[NumDoc] is not null --and c.marca  IN ('PPD','PPD PLUS')    
--and c.RETIRO is null    
  
  
/*=================================================================================================================*/    
    
Select row_number() OVER (PARTITION BY Llave_X_Dia ORDER BY CONVERT(INT,Pesos) ASC) AS Mejor_Llam_Dia , * INTO tmp_gestion_diners_ppd_302 FROM tmp_gestion_diners_ppd_301 with (nolock);    
Select row_number() OVER (PARTITION BY llave_x_Mes ORDER BY CONVERT(INT,Pesos) asc,Fecha_Llamada Desc) AS Mejor_Llam_Mes , * INTO tmp_gestion_diners_ppd_303 FROM tmp_gestion_diners_ppd_302 with (nolock);    
/*=================================================================================================================*/    
    
Declare @Cadena2 VARCHAR(Max)    
Declare @FinaQuery2 VARCHAR(Max)    
    
Set @Cadena2 =    
'    
SELECT a.*,b.tMontoF7,c.tApePaterno,c.tApeMaterno,c.tNombres,b.tTipoCampanaF7     
FROM bdsae.tb_clientes a, bdsae.tb_solicitud b, bdbbva.usuario c    
where a.tRegistro =b.tRegistro and c.tCodigo =a.tUsuario    
and date(a.fFechaGrabacion)=''''' +  CONVERT(varchar(10), @FECHAINI, 23) + '''''    
'    
Set @FinaQuery2 ='SELECT * Into vnts_diners_1 FROM OPENQUERY([192.168.3.8],' + '''' +@Cadena2 + '''' + ')'    
Exec(@FinaQuery2);    
/*=================================================================================================================*/    
    
--SELECT * FROM vnts_diners_1 WHERE tDocumento='08259463'    
    
Update a1    
Set a1.[Ventas]=0    
From tmp_gestion_diners_ppd_303 a1,tmp_gestion_diners_ppd_303 a2    
Where a1.mejor_llam_mes > a2.mejor_llam_mes    
And a1.DNI_cliente = a2.DNI_cliente;    
    
Update a1    
Set a1.[ESTADO VALIDACION]=''    
From tmp_gestion_diners_ppd_303 a1,tmp_gestion_diners_ppd_303 a2    
Where a1.mejor_llam_mes > a2.mejor_llam_mes    
And a1.DNI_cliente = a2.DNI_cliente;    
    
Update a1    
Set a1.Monto_Abonar=0    
From tmp_gestion_diners_ppd_303 a1,tmp_gestion_diners_ppd_303 a2    
Where a1.mejor_llam_mes > a2.mejor_llam_mes    
And a1.DNI_cliente = a2.DNI_cliente;    
/*=================================================================================================================*/    
    
Select * Into tmp_gestion_diners_ppd_304 from tmp_gestion_diners_ppd_303 --where Mejor_Llam_Mes=1;    
ALTER TABLE tmp_gestion_diners_ppd_304 ALTER COLUMN Monto_Abonar numeric(18,2) NULL    
/*=================================================================================================================*/    
    
update a    
set    
a.data_banco='CONTACTO EFECTIVO>>VENTA>>SOCIO ACEPTO OFERTA PPD',    
a.Monto_Abonar=b.tMontoF7,    
a.INDICE='1'    
from tmp_gestion_diners_ppd_304 a inner join vnts_diners_1 b on a.Dni_Cliente=b.tDocumento COLLATE Modern_Spanish_CI_AS    
where Mejor_Llam_Mes=1 AND b.tTipoCampanaF7 in ('PPD','PPD PLUS')    
    
update a    
set    
a.data_banco='CONTACTO EFECTIVO>>VENTA>>SOCIO ACEPTO OFERTA BT',    
a.Monto_Abonar=b.tMontoF7,    
a.INDICE='1'    
from tmp_gestion_diners_ppd_304 a inner join vnts_diners_1 b on a.Dni_Cliente=b.tDocumento COLLATE Modern_Spanish_CI_AS    
where Mejor_Llam_Mes=1 AND b.tTipoCampanaF7 in ('CD')     
    
update a    
set    
a.data_banco='CONTACTO EFECTIVO>>VENTA>>SOCIO ACEPTO OFERTA DISEF',    
a.Monto_Abonar=b.tMontoF7,    
a.INDICE='1'    
from tmp_gestion_diners_ppd_304 a inner join vnts_diners_1 b on a.Dni_Cliente=b.tDocumento COLLATE Modern_Spanish_CI_AS    
where Mejor_Llam_Mes=1 AND b.tTipoCampanaF7 in ('DE')    
    
    
--select * from tmp_gestion_diners_ppd_304 where Dni_Cliente='06255250'    
    
/*=================================================================================================================*/    
    
update a    
set    
a.data_banco='CONTACTO EFECTIVO>>VOLVER A LLAMAR>>TITULAR DESEA QUE VUELVAN A LLAMAR',    
INDICE='22'    
from tmp_gestion_diners_ppd_304 a where a.INDICE='1' and a.Monto_Abonar is null    
    
update a    
set    
a.data_banco='CONTACTO EFECTIVO>>VOLVER A LLAMAR>>TITULAR DESEA QUE VUELVAN A LLAMAR',    
INDICE='22',    
Monto_Abonar= null    
from tmp_gestion_diners_ppd_304 a where a.INDICE='1' and a.Monto_Abonar=0     
    
update a    
set    
Monto_Abonar= null    
from tmp_gestion_diners_ppd_304 a where a.Monto_Abonar=0     
    
/*=================================================================================================================*/    
    
    
    
DELETE FROM Historial_Target_Diners_PPD WHERE CONVERT(DATE,FECHA_INICIO_GESTION)=@FECHAINI    
    
Insert INTO Historial_Target_Diners_PPD    
SELECT distinct    
Dni_Cliente AS DNI,    
Telefono_Llamado AS TELEFONO,    
INDICE AS ID,    
data_banco AS CLASIFICACION,    
'TARGET' AS 'CALL',    
'1' AS 'NUMERO_GESTIONES_REALIZADAS',    
SUBSTRING(CONVERT(varchar,a.Inicio),1,19)  AS 'FECHA_INICIO_GESTION',    
isnull(SUBSTRING(CONVERT(varchar,a.Fin),1,19),SUBSTRING(CONVERT(varchar,a.Inicio),1,19)) AS 'FECHA_FIN_LLAMADA',    
SUBSTRING(CONVERT(varchar,a.Fecha_Hora_Llamada),1,19)  AS 'FECHA_HORA_MEJOR_GESTION',    
case when Estados='CONTACTO EFECTIVO' then SUBSTRING(CONVERT(VARCHAR,TMO),4,2)+' minuto(s)'+' '+SUBSTRING(CONVERT(VARCHAR,TMO),7,2)+' segundo(s)' end as 'DURACION_LLAMADA_EFECTIVA',    
case when Estados='NO CONTACTO' then SUBSTRING(CONVERT(VARCHAR,TMO),4,2)+' minuto(s)'+' '+SUBSTRING(CONVERT(VARCHAR,TMO),7,2)+' segundo(s)' end as 'DURACION_LLAMADA_NO_EFECTIVA',    
case when ejecutivo='Outbound Auto Dial' THEN 'MARCADOR' ELSE ejecutivo END as 'EJECUTIVO',    
isnull(C.tDni,'MARCADOR') AS 'DNI_EJECUTIVO',--DNI_Ejecutivo AS 'DNI_EJECUTIVO',    
NULL AS OBSERVCION_GESTION,    
Monto_Abonar as 'MONTO_VENDIDO',    
A.BASE AS TIPO_BD    
FROM tmp_gestion_diners_ppd_304 A     
INNER JOIN cont_gestion_diners_ppd B ON A.Dni_Cliente=B.Dni     
LEFT JOIN [URANO].[dbo].[tPersonalCrm] C ON C.tLogin = A.DNI_Ejecutivo 
order by Monto_Abonar desc    
  
--SELECT * FROM [URANO].[dbo].[tPersonalCrm]   
/*=================================================================================================================*/    
--UPDATE a     
--SET a.TIPO_BD='PPD-BT'     
--FROM Historial_Target_Diners_PPD a LEFT JOIN [ODIN].[dbo].Base_Maestra_Diners_Vigente b ON a.DNI=b.[NumDoc] WHERE  B.FECHA_ENVIO IN ('2024-01-17')    
    
--UPDATE a     
--SET a.TIPO_BD='PPD PLUS'     
--FROM Historial_Target_Diners_PPD a LEFT JOIN [ODIN].[dbo].Base_Maestra_Diners_Vigente b ON a.DNI=b.[NumDoc] WHERE  B.FECHA_ENVIO IN ('2024-01-10','2024-01-18')    
    
--SELECT * FROM [ODIN].[dbo].Base_Maestra_Diners_Vigente WHERE  FECHA_ENVIO IN ('2024-01-17')    
    
If object_id('tmp_gestion_diners_ppd_300') is not null    
Drop table tmp_gestion_diners_ppd_300;    
If object_id('cont_gestion_diners_ppd') is not null    
Drop table cont_gestion_diners_ppd;    
If object_id('tmp_gestion_diners_ppd_301') is not null    
Drop table tmp_gestion_diners_ppd_301;    
If object_id('tmp_gestion_diners_ppd_302') is not null    
Drop table tmp_gestion_diners_ppd_302;    
If object_id('tmp_gestion_diners_ppd_303') is not null    
Drop table tmp_gestion_diners_ppd_303;    
If object_id('tmp_gestion_diners_ppd_304') is not null    
Drop table tmp_gestion_diners_ppd_304;    
If object_id('vnts_diners_1') is not null     
drop table vnts_diners_1;    
/*=================================================================================================================*/    
truncate table [tb_reporte_feedback_Target_Diners_PP]    
    
insert into [tb_reporte_feedback_Target_Diners_PP]    
select DNI, TELEFONO, ID, CLASIFICACION, CALL, NUMERO_GESTIONES_REALIZADAS, FECHA_INICIO_GESTION,    
FECHA_FIN_LLAMADA, FECHA_HORA_MEJOR_GESTION, isnull(DURACION_LLAMADA_EFECTIVA,''), isnull(DURACION_LLAMADA_NO_EFECTIVA,''),    
EJECUTIVO, DNI_EJECUTIVO, isnull(OBSERVCION_GESTION,''), isnull(MONTO_VENDIDO,''), TIPO_BD from Historial_Target_Diners_PPD     
where convert(date,FECHA_INICIO_GESTION)=@FECHAINI order by convert(int,id);    
    
  
  
  
  
  

## tc

In [ ]:


DROP TABLE IF EXISTS tmp_gestion_diners_tc_300;
DROP TABLE IF EXISTS cont_gestion_diners_tc;
DROP TABLE IF EXISTS tmp_gestion_diners_tc_301;
DROP TABLE IF EXISTS tmp_gestion_diners_tc_302;
DROP TABLE IF EXISTS tmp_gestion_diners_tc_303;
DROP TABLE IF EXISTS tmp_gestion_diners_tc_304;

DECLARE @FECHAINI date = '2026-07-13';
DECLARE @CAMPANA varchar(50) = FORMAT(GETDATE(), 'yyyy-MM') + ' DINERS TC';
/*=============================================================================================================================*/

Select dni+' '+PHONE_NUMBER AS Llave_Primaria,* 
into tmp_gestion_diners_tc_300
From [dbo].[tmp_llamadas_mes] 
where Fecha_Llamada=@FECHAINI and 
Nombre_Campana IN (@CAMPANA) 
/*=============================================================================================================================*/
Select 
C.FECHA_ENVIO,
'Diners' AS Origen_Registro,
Case 
When a.Nombre_Campana IN (@CAMPANA) Then 'Diners Club' Else 'Mal' End As Servicio,
convert(varchar,C.SERVICIO)+' '+a.Dni+' '+Convert(Varchar,a.Fecha_Llamada,103) as Llave_X_Dia,
convert(varchar,C.SERVICIO)+' '+a.Dni as Llave_X_Mes,
a.Dni As Dni_Cliente,
case when a.DNI_Ejecutivo='VDAD' THEN 'MARCADOR' else f.[tDni] end as DNI_Ejecutivo,
case when a.Ejecutivo='Outbound Auto Dial' then 'MARCADOR' else a.Ejecutivo end as Ejecutivo,
a.Fecha_Hora_Llamada,
a.Fecha_Llamada,
a.Trama_Hora,
a.PHONE_NUMBER as Telefono_Llamado,
ISNULL(C.MARCA,'SP') as [ESTADO VALIDACION],
Case When e.[NIVEL 3] = 'VENTA' Then 1 Else '' End AS Ventas,
Case When e.[NIVEL 3] = 'VENTA' Then c.[LÍNEA CRÉDITO DOLARES] Else '' End AS Monto_Abonar,
a.Codigo_Paleta,
e.peso as Pesos,
e.nCodCliente,
1 as Cantidad,
e.[NIVEL 2] as Estados,
e.nCodCliente AS INDICE,
e.[NIVEL 2]+'>>'+e.[NIVEL 3]+'>>'+e.[NIVEL 4] AS data_banco,
CONVERT(nvarchar(8),DATEADD(SECOND,segundos,0),114) as [TMO],
a.Inicio,a.Fin,a.Comentarios
Into tmp_gestion_diners_tc_301
from tmp_gestion_diners_tc_300 a with (nolock) 
LEFT Join ODIN..BASE_MAESTRA_DINERS_TC_VIGENTE C with (nolock) on a.Dni=C.NUMERO_DOCUMENTO
Left Join ODIN..tNumeroDinersTc d with (nolock) on a.Llave_Primaria=d.Llave
LEFT JOIN ODIN..tTipologia_Diners_TC e ON a.Codigo_Paleta=e.cod
LEFT JOIN [URANO].[dbo].[tPersonalCrm] f on a.DNI_Ejecutivo=f.[tLogin]
where C.SERVICIO in ('02','2') and (c.RETIRO is null or c.RETIRO = '') and e.DETALLE is not null;


Select row_number() OVER (PARTITION BY Llave_X_Dia ORDER BY CONVERT(INT,Pesos) ASC) AS Mejor_Llam_Dia , * INTO tmp_gestion_diners_tc_302 FROM tmp_gestion_diners_tc_301 with (nolock);

Select row_number() OVER (PARTITION BY llave_x_Mes ORDER BY CONVERT(INT,Pesos) asc,Fecha_Llamada Desc) AS Mejor_Llam_Mes , * INTO tmp_gestion_diners_tc_303 FROM tmp_gestion_diners_tc_302 with (nolock);
/*=============================================================================================================================*/

If object_id('VENTAS_DINERSTC') is not null drop table VENTAS_DINERSTC  

Declare @Cadena2 VARCHAR(Max)  
Declare @FinaQuery2 VARCHAR(Max)  
  
Set @Cadena2 =  
'SELECT a.*,c.tApePaterno,c.tApeMaterno,c.tNombres   
FROM bddinerstc.tb_clientes a, bddinerstc.tb_solicitud b, bdbbva.usuario c  
where a.tRegistro =b.tRegistro and c.tCodigo =a.tUsuario  
and date(a.fFechaGrabacion)=''''' + CONVERT(varchar(10), @FECHAINI, 23)  + '''''
'
Set @FinaQuery2 ='SELECT * Into VENTAS_DINERSTC FROM OPENQUERY([192.168.3.8],' + '''' +@Cadena2 + '''' + ')'  
Exec(@FinaQuery2);
/*=============================================================================================================================*/

Update a1
Set a1.[Ventas]=0
From tmp_gestion_diners_tc_303 a1,tmp_gestion_diners_tc_303 a2
Where a1.mejor_llam_mes > a2.mejor_llam_mes
And a1.DNI_cliente = a2.DNI_cliente;

Update a1
Set a1.[ESTADO VALIDACION]=''
From tmp_gestion_diners_tc_303 a1,tmp_gestion_diners_tc_303 a2
Where a1.mejor_llam_mes > a2.mejor_llam_mes
And a1.DNI_cliente = a2.DNI_cliente;

Update a1
Set a1.Monto_Abonar=0
From tmp_gestion_diners_tc_303 a1,tmp_gestion_diners_tc_303 a2
Where a1.mejor_llam_mes > a2.mejor_llam_mes
And a1.DNI_cliente = a2.DNI_cliente;

Select * Into tmp_gestion_diners_tc_304 from tmp_gestion_diners_tc_303; --where Mejor_Llam_Mes=1;
/*=============================================================================================================================*/
DELETE FROM Historial_Target_Diners_TC WHERE CONVERT(DATE,FECHA_INICIO_GESTION)=@FECHAINI

Insert Into Historial_Target_Diners_TC
SELECT distinct
Dni_Cliente AS DNI,
Telefono_Llamado AS TELEFONO,
INDICE AS ID,
data_banco AS CLASIFICACION,
'TARGET' AS 'CALL',
'1' AS 'NUMERO_GESTIONES_REALIZADAS',
SUBSTRING(CONVERT(varchar,a.Inicio),1,19)  AS 'FECHA_INICIO_GESTION',
isnull(SUBSTRING(CONVERT(varchar,a.Fin),1,19),SUBSTRING(CONVERT(varchar,a.Inicio),1,19)) AS 'FECHA_FIN_LLAMADA',
case when Estados='CONTACTO EFECTIVO' then SUBSTRING(CONVERT(VARCHAR,TMO),4,2)+' minuto(s)'+' '+SUBSTRING(CONVERT(VARCHAR,TMO),7,2)+' segundo(s)' end as 'DURACION_LLAMADA_EFECTIVA',
case when Estados='NO CONTACTO' then SUBSTRING(CONVERT(VARCHAR,TMO),4,2)+' minuto(s)'+' '+SUBSTRING(CONVERT(VARCHAR,TMO),7,2)+' segundo(s)' end as 'DURACION_LLAMADA_NO_EFECTIVA',
ejecutivo as 'EJECUTIVO',
DNI_Ejecutivo AS 'DNI_EJECUTIVO',
LTRIM(RTRIM(A.Comentarios)) --CAST(NULL AS VARCHAR(20)) 
AS 'FLUJO_INTERESADO'
FROM tmp_gestion_diners_tc_304 A 




In [ ]:
ALTER PROCEDURE [dbo].[ALFIN_FEEDBACK] (@CAMPAÑA_ALFIN VARCHAR(20), @DIA INT) AS



WITH LLAMADAS_TC AS (
    SELECT
        IDX = ROW_NUMBER() OVER(PARTITION BY g.Dni_cliente ORDER BY tip.id_banco ASC),
        g.dni_cliente AS DNI,
        REPLACE(CONVERT(varchar, CONVERT(varchar, g.Fecha, 23), 112), '-', '') AS FECHA_GESTION,
        CONVERT(varchar, g.hora, 8) AS HORA_GESTION,
        g.celular AS TELEFONO,
        tip.ID_BANCO AS COD_TIPO,
        ISNULL(p.dni, g.id_personal) AS DNI_ASESOR,
        REPLACE(CONVERT(varchar, c.CL_CARGA, 112), '-', '') AS FECHA_ENVIO,
        c.campania AS COD_CAMPANA, c.OFERTA_MAX AS OFERTA
    FROM alfin_gestion g
    LEFT JOIN alfin_clientes c ON g.dni_cliente = c.NUMERO_DOCUMENTO
    LEFT JOIN personal p ON p.dni = g.dni_ejecutivo
    LEFT JOIN alfin_tipificaciones tip ON tip.id = g.tipificacion
	inner join servicios s on g.servicio=s.id and c.cl_base=s.base and g.Campaña=s.base
    WHERE g.fecha = CONVERT(DATE, GETDATE() - @DIA, 103)
    AND tip.estado = 'A'
)
SELECT
    'TAG202' AS COD_CANAL, 'TARGET' AS CANAL, DNI, FECHA_ENVIO, FECHA_GESTION, HORA_GESTION,
    TELEFONO, 'T1' AS ORIGEN_TELEFONO, COD_CAMPANA, COD_TIPO, OFERTA,
    CASE WHEN DNI_ASESOR = '99999999' THEN '' ELSE DNI_ASESOR END AS DNI_ASESOR
FROM LLAMADAS_TC


In [1]:

import sys 
sys.path.append('C:/Users/DATA/Documents/datos/01_script/inicio/funciones')
from funciones import *
from funciones_spark import *
from variables_inicio import *
from utils_sql import *
from unidecode import unidecode
from sqlalchemy import create_engine
from sqlalchemy import text

fecha_mes_base='2026-06-01'
tipi_cond1='RECLUTAMIENTO'
servidor_01=64

campana='negocios'

server_sql = server_kishin
db_sql = "DANTALION"
user_sql = user_kishin
pwd_sql = pwd_kishin

engine_kishin = create_engine(
    f"mssql+pyodbc://{user_sql}:{pwd_sql}@{server_sql}/{db_sql}"
    "?driver=ODBC+Driver+17+for+SQL+Server"
)
server_sql = server_zeus
db_sql = "ODIN"
user_sql = user_zeus
pwd_sql = pwd_zeus
engine_zeus = create_engine(
    f"mssql+pyodbc://{user_sql}:{pwd_sql}@{server_sql}/{db_sql}"
    "?driver=ODBC+Driver+17+for+SQL+Server"
)
server_sql = server_sa
db_sql = "ODIN"
user_sql = user_sa
pwd_sql = pwd_sa
engine_sa = create_engine(
    f"mssql+pyodbc://{user_sql}:{pwd_sql}@{server_sql}/{db_sql}"
    "?driver=ODBC+Driver+17+for+SQL+Server"
)
server_sql = server_zeus
db_sql = "SAMANTHA"
user_sql = user_zeus
pwd_sql = pwd_zeus
engine_samantha = create_engine(
    f"mssql+pyodbc://{user_sql}:{pwd_sql}@{server_sql}/{db_sql}"
    "?driver=ODBC+Driver+17+for+SQL+Server"
)

server_sql = server_sa
db_sql = "VALENTINA"
user_sql = user_sa
pwd_sql = pwd_sa
engine_vale = create_engine(
    f"mssql+pyodbc://{user_sql}:{pwd_sql}@{server_sql}/{db_sql}"
    "?driver=ODBC+Driver+17+for+SQL+Server"
)

### Alfin

In [15]:
fecha_gestion = "2026-06-28"

query = f"""
    WITH LLAMADAS_TC AS (
        SELECT
            ROW_NUMBER() OVER(
                PARTITION BY g.Dni_cliente 
                ORDER BY tip.id_banco ASC
            ) AS IDX,
            g.dni_cliente AS DNI,
            CONVERT(VARCHAR(8), g.Fecha, 112) AS FECHA_GESTION,
            CONVERT(VARCHAR(8), g.hora, 108) AS HORA_GESTION,
            g.celular AS TELEFONO,
            tip.ID_BANCO AS COD_TIPO,
            ISNULL(p.dni, g.id_personal) AS DNI_ASESOR,
            CONVERT(VARCHAR(8), c.CL_CARGA, 112) AS FECHA_ENVIO,
            c.campania AS COD_CAMPANA,
            c.OFERTA_MAX AS OFERTA
        FROM valentina.dbo.alfin_gestion g
        LEFT JOIN valentina.dbo.alfin_clientes c 
            ON g.dni_cliente = c.NUMERO_DOCUMENTO
        LEFT JOIN valentina.dbo.personal p 
            ON p.dni = g.dni_ejecutivo
        LEFT JOIN valentina.dbo.alfin_tipificaciones tip 
            ON tip.id = g.tipificacion
        INNER JOIN valentina.dbo.servicios s 
            ON g.servicio = s.id
            AND c.cl_base = s.base
            AND g.Campaña = s.base
        WHERE g.fecha = '{fecha_gestion}'
        AND tip.estado = 'A'
    )
    SELECT
        'TAG202' AS COD_CANAL,
        'TARGET' AS CANAL,
        DNI,
        FECHA_ENVIO,
        FECHA_GESTION,
        HORA_GESTION,
        TELEFONO,
        'T1' AS ORIGEN_TELEFONO,
        COD_CAMPANA,
        COD_TIPO,
        OFERTA,
        CASE 
            WHEN DNI_ASESOR = '99999999' THEN '' 
            ELSE DNI_ASESOR 
        END AS DNI_ASESOR
    FROM LLAMADAS_TC
    """

df_feedback_target = pd.read_sql(query, engine_vale)
df_feedback_target.head()


,COD_CANAL,CANAL,DNI,FECHA_ENVIO,FECHA_GESTION,HORA_GESTION,TELEFONO,ORIGEN_TELEFONO,COD_CAMPANA,COD_TIPO,OFERTA,DNI_ASESOR


,COD_CANAL,CANAL,DNI,FECHA_ENVIO,FECHA_GESTION,HORA_GESTION,TELEFONO,ORIGEN_TELEFONO,COD_CAMPANA,COD_TIPO,OFERTA,DNI_ASESOR


In [31]:
fecha_gestion = "2026-06-23"

query = f"""
    WITH LLAMADAS_TC AS (
        SELECT
            ROW_NUMBER() OVER(
                PARTITION BY g.Dni_cliente 
                ORDER BY tip.id_banco ASC
            ) AS IDX,
            g.dni_cliente AS DNI,
            CONVERT(VARCHAR(8), g.Fecha, 112) AS FECHA_GESTION,
            CONVERT(VARCHAR(8), g.hora, 108) AS HORA_GESTION,
            g.celular AS TELEFONO,
            tip.ID_BANCO AS COD_TIPO,
            ISNULL(p.dni, g.id_personal) AS DNI_ASESOR,
            CONVERT(VARCHAR(8), c.CL_CARGA, 112) AS FECHA_ENVIO,
            'credicash' AS COD_CAMPANA,
            c.OFERTA_CREDICASH AS OFERTA,
            '' AS SUB_DESCRIPCION,
            '' AS DESCRIPCION
        FROM valentina.dbo.alfcc_gestion g
        LEFT JOIN valentina.dbo.alfcc_clientes c 
            ON g.dni_cliente = c.NUMERO_DOCUMENTO
        LEFT JOIN valentina.dbo.personal p 
            ON p.dni = g.dni_ejecutivo
        LEFT JOIN valentina.dbo.alfcc_tipificaciones tip 
            ON tip.id_banco = g.tipificacion
        INNER JOIN valentina.dbo.servicios s 
            ON g.servicio = s.id
        AND c.cl_base = s.base
        AND g.Campaña = s.base
        WHERE g.fecha = '{fecha_gestion}'
        AND tip.estado = 'A'
    )
    SELECT
        'TAGCASH' AS COD_CANAL,
        'TARGET' AS CANAL,
        DNI,
        FECHA_ENVIO,
        FECHA_GESTION,
        HORA_GESTION,
        TELEFONO,
        '' AS ORIGEN_TELEFONO,
        COD_CAMPANA,
        COD_TIPO,
        SUB_DESCRIPCION,
        DESCRIPCION,
        OFERTA,
        CASE 
            WHEN DNI_ASESOR = '99999999' THEN '' 
            ELSE DNI_ASESOR 
        END AS DNI_ASESOR
    FROM LLAMADAS_TC
"""

df_feedback_target = pd.read_sql(query, engine_vale)

In [32]:
df_feedback_target.shape

(22703, 14)

In [33]:
fecha=23

In [34]:
ruta_archivo = rf"\\192.168.2.30\hiroom2\SQLServer Compartido\05.FEEDBACKS\CREDICASH_ALFIN\FEEDBACK_TARGET_202606{fecha}.csv"

In [35]:
df_feedback_target.to_csv(
    ruta_archivo,
    sep="|",
    index=False,
    encoding="utf-8"
)

In [10]:
print(ruta_archivo)

\\192.168.2.30\hiroom2\SQLServer Compartido\05.FEEDBACKS\CREDICASH_ALFIN\FEEDBACK_TARGET_24.csv


In [7]:
df_feedback_target.head()

,COD_CANAL,CANAL,DNI,FECHA_ENVIO,FECHA_GESTION,HORA_GESTION,TELEFONO,ORIGEN_TELEFONO,COD_CAMPANA,COD_TIPO,SUB_DESCRIPCION,DESCRIPCION,OFERTA,DNI_ASESOR
0,TAGCASH,TARGET,01040241,20260622,20260624,11:26:37,983154874,,credicash,27,,,2500,
1,TAGCASH,TARGET,01041191,20260622,20260624,12:04:18,980534249,,credicash,27,,,7000,
2,TAGCASH,TARGET,01081174,20260622,20260624,13:44:13,990072843,,credicash,27,,,12400,
3,TAGCASH,TARGET,01121020,20260622,20260624,12:47:02,951441451,,credicash,27,,,6400,
4,TAGCASH,TARGET,01121176,20260622,20260624,12:20:00,992899296,,credicash,27,,,14000,


In [ ]:
ALTER PROCEDURE [dbo].[ALFIN_FEEDBACK] (@CAMPAÑA_ALFIN VARCHAR(20), @DIA INT) AS
